# 可选实验：特征缩放与学习率（多变量）

## 目标
在本实验中，您将：
- 使用上一个实验中开发的多变量例程；
- 在具有多个特征的数据集上运行梯度下降；
- 探索*学习率 alpha* 对梯度下降的影响；
- 使用 z-score 归一化进行*特征缩放*，以改善梯度下降的性能。

## 工具
你将使用上一个实验中开发的函数，以及 Matplotlib 和 NumPy。

In [ ]:
import numpy as np
np.set_printoptions(precision=2)
import matplotlib.pyplot as plt
dlblue = '#0096ff'; dlorange = '#FF9300'; dldarkred='#C00000'; dlmagenta='#FF40FF'; dlpurple='#7030A0'; 
plt.style.use('./deeplearning.mplstyle')
from lab_utils_multi import  load_house_data, compute_cost, run_gradient_descent 
from lab_utils_multi import  norm_plot, plt_contour_multi, plt_equal_scale, plot_cost_i_w

## 记号

|通用 <img width=70/> <br />  记号  <img width=70/> | 描述<img width=350/>| Python（如适用） |
|: ------------|: ------------------------------------------------------------||
| $a$ | 标量，非粗体                                                      ||
| $\mathbf{a}$ | 向量，粗体                                                 ||
| $\mathbf{A}$ | 矩阵，大写粗体                                         ||
| **回归** |         |    |     |
|  $\mathbf{X}$ | 训练样本矩阵                  | `X_train` |
|  $\mathbf{y}$  | 训练样本目标值                | `y_train`
|  $\mathbf{x}^{(i)}$, $y^{(i)}$ | 第 $i_{th}$ 个训练样本 | `X[i]`, `y[i]`|
| m | 训练样本数量 | `m`|
| n | 每个样本中的特征数量 | `n`|
|  $\mathbf{w}$  | 参数：权重，                       | `w`    |
|  $b$           | 参数：偏置                                           | `b`    |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | 在由 $\mathbf{w},b$ 参数化的 $\mathbf{x}^{(i)}$ 处的模型评估结果：$f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)}+b$  | `f_wb` |
|$\frac{\partial J(\mathbf{w},b)}{\partial w_j}$| 代价相对于参数 $w_j$ 的梯度或偏导数 |`dj_dw[j]`|
|$\frac{\partial J(\mathbf{w},b)}{\partial b}$| 代价相对于参数 $b$ 的梯度或偏导数| `dj_db`|

# 问题陈述

与此前实验一样，你将使用房价预测这一示例。训练数据集包含许多具有 4 个特征（面积、卧室数、楼层数和房龄）的样本，如下表所示。请注意，在本实验中，面积特征的单位是平方英尺，而此前实验使用的是 1000 平方英尺。这个数据集比此前实验的数据集更大。

我们希望使用这些值构建一个线性回归模型，然后便能预测其他房屋的价格，例如一栋面积为 1200 平方英尺、有 3 间卧室、1 层楼、房龄 40 年的房屋。

## 数据集：
| 面积（平方英尺） | 卧室数量  | 楼层数量 | 房龄 | 价格（千美元）  |   
| ----------------| ------------------- |----------------- |--------------|----------------------- |  
| 952             | 2                   | 1                | 65           | 271.5                  |  
| 1244            | 3                   | 2                | 64           | 232                    |  
| 1947            | 3                   | 2                | 17           | 509.8                  |  
| ...             | ...                 | ...              | ...          | ...                    |

In [ ]:
# load the dataset
X_train, y_train = load_house_data()
X_features = ['size(sqft)','bedrooms','floors','age']

通过分别绘制每个特征与价格的关系图，来查看数据集及其特征。

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000's)")
plt.show()

分别绘制每个特征与目标值（价格）的关系图，可以在一定程度上看出哪些特征对价格影响最大。从上图可见，面积增加时价格也会上升。卧室数量和楼层数似乎对价格没有太大影响。较新的房屋比老旧房屋价格更高。

<a name="toc_15456_5"></a>
## 多变量梯度下降
以下是你在上一个多变量梯度下降实验中推导出的公式：

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline\;
& w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{for j = 0..n-1}\newline
&b\ \ := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

其中，n 是特征数，参数 $w_j$ 和 $b$ 会同时更新，并且

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{2}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{3}
\end{align}
$$
* m 是数据集中的训练样本数

    
* $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是模型的预测值，而 $y^{(i)}$ 是目标值

## 学习率
<figure>
    <img src="./images/C1_W2_Lab06_learningrate.PNG" style="width:1200px;" >
</figure>
课程讨论了设置学习率 $\alpha$ 时涉及的一些问题。学习率控制参数更新的步长，参见上面的公式 (1)。所有参数共用同一个学习率。

让我们运行梯度下降，并在数据集上尝试 $\alpha$ 的几种设置

### $\alpha$ = 9.9e-7

In [ ]:
#set alpha to 9.9e-7
_, _, hist = run_gradient_descent(X_train, y_train, 10, alpha = 9.9e-7)

学习率似乎过高。解无法收敛。代价不降反而在*上升*。让我们绘制结果：

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

右图显示了其中一个参数 $w_0$ 的值。每次迭代时，它都会越过最优值，因此代价最终不降反而*上升*，而不是逐渐接近最小值。请注意，这幅图并不完全准确，因为每一轮实际修改的是 4 个参数，而非只有一个。图中仅显示 $w_0$，其他参数则固定在合适的值。在这幅图和后面的图中，你可能会注意到蓝线与橙线略有偏差。


### $\alpha$ = 9e-7
让我们尝试稍小一点的值，看看会发生什么。

In [ ]:
#set alpha to 9e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 9e-7)

代价在整个运行过程中不断减小，表明 alpha 并未过大。

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

在左图中，可以看到代价正如预期的那样不断减小。在右图中，可以看到 $w_0$ 仍在最小值附近振荡，但它在每次迭代中都在减小，而不是增大。请注意，在上图中，随着 `w[0]` 越过最优值，`dj_dw[0]` 在每次迭代时都会改变符号。
这个 alpha 值将会收敛。你可以改变迭代次数，观察其表现。

### $\alpha$ = 1e-7
让我们为 $\alpha$ 尝试一个稍小的值，看看会发生什么。

In [ ]:
#set alpha to 1e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 1e-7)

代价在整个运行过程中持续下降，表明 $\alpha$ 并没有过大。

In [ ]:
plot_cost_i_w(X_train,y_train,hist)

左图中可以看到，代价正如预期那样下降。右图中可以看到，$w_0$ 在下降，但没有越过最小值。请注意上方的 `dj_w0` 在整个运行过程中都为负值。该解也会收敛，只是速度不如上一个示例快。

## 特征缩放
<figure>
    <img src="./images/C1_W2_Lab06_featurescalingheader.PNG" style="width:1200px;" >
</figure>
课程说明了对数据集进行重新缩放、使特征具有相近取值范围的重要性。
如果对背后的详细原因感兴趣，请点击下面的“详细信息”标题；否则，下一节将逐步介绍如何实现特征缩放。

<details>
<summary>
    <font size='3', color='darkgreen'><b>详细信息</b></font>
</summary>

让我们再次观察 $\alpha$ = 9e-7 时的情况。这个值已经非常接近 $\alpha$ 在不发散的情况下可设置的最大值。下面是一段简短的运行结果，展示了前几次迭代：

<figure>
    <img src="./images/C1_W2_Lab06_ShortRun.PNG" style="width:1200px;" >
</figure>

上面可以看到，虽然代价正在下降，但显然 $w_0$ 的进展比其他参数更快，这是因为它的梯度要大得多。

下图展示了使用 $\alpha$ = 9e-7 进行一次超长时间运行的结果。这需要数小时。

<figure>
    <img src="./images/C1_W2_Lab06_LongRun.PNG" style="width:1200px;" >
</figure>
    
上面可以看到，代价在最初下降后便缓慢降低。请注意 `w0` 与 `w0`、`w1`、`w2` 之间的差异，以及 `dj_dw0` 与 `dj_dw1-3` 之间的差异。`w0` 很快就接近其最终值，而 `dj_dw0` 也迅速减小到一个很小的值，这表明 `w0` 已接近最终值。其他参数则下降得慢得多。

这是为什么？有什么可以改进的地方吗？请看下面：
<figure>
    <center> <img src="./images/C1_W2_Lab06_scale.PNG"   ></center>
</figure>   

上图说明了为什么各个 $w$ 的更新并不均衡。
- 所有参数更新（$w$ 和 $b$）都共享 $\alpha$。
- 对于各个 $w$，公共误差项还要乘以相应特征（$b$ 不需要）。
- 各特征的量级差异显著，导致某些特征的更新速度远快于其他特征。在本例中，$w_0$ 乘以“面积（平方英尺）”，其值通常 > 1000；而 $w_1$ 乘以“卧室数量”，其值通常为 2～4。
    
解决办法是进行特征缩放。

课程讨论了三种不同技术：
- 特征缩放：本质上是将每个特征除以用户选择的值，使其范围落在 -1 到 1 之间。
- 均值归一化：$x_i := \dfrac{x_i - \mu_i}{max - min} $
- Z-score 归一化，我们将在下面进行探索。


### z-score 归一化
经过 z-score 归一化后，所有特征的均值都将为 0，标准差都将为 1。

要实现 z-score 归一化，请按以下公式调整输入值：
$$x^{(i)}_j = \dfrac{x^{(i)}_j - \mu_j}{\sigma_j} \tag{4}$$ 
其中，$j$ 选择 X 矩阵中的一个特征或一列。$µ_j$ 是特征 (j) 所有值的均值，$\sigma_j$ 是特征 (j) 的标准差。
$$
\begin{align}
\mu_j &= \frac{1}{m} \sum_{i=0}^{m-1} x^{(i)}_j \tag{5}\\
\sigma^2_j &= \frac{1}{m} \sum_{i=0}^{m-1} (x^{(i)}_j - \mu_j)^2  \tag{6}
\end{align}
$$

>**实现说明：** 对特征进行归一化时，务必保存归一化所使用的值，即计算中使用的均值和标准差。从模型中学习得到参数后，我们经常希望预测此前未见过的房屋价格。给定一个新的 x 值（客厅面积和卧室数量），必须先使用此前从训练集中计算得到的均值和标准差对 x 进行归一化。

**实现**

In [ ]:
def zscore_normalize_features(X):
    """
    computes  X, zcore normalized by column
    
    Args:
      X (ndarray): Shape (m,n) input data, m examples, n features
      
    Returns:
      X_norm (ndarray): Shape (m,n)  input normalized by column
      mu (ndarray):     Shape (n,)   mean of each feature
      sigma (ndarray):  Shape (n,)   standard deviation of each feature
    """
    # find the mean of each column/feature
    mu     = np.mean(X, axis=0)                 # mu will have shape (n,)
    # find the standard deviation of each column/feature
    sigma  = np.std(X, axis=0)                  # sigma will have shape (n,)
    # element-wise, subtract mu for that column from each example, divide by std for that column
    X_norm = (X - mu) / sigma      

    return (X_norm, mu, sigma)
 
#check our work
#from sklearn.preprocessing import scale
#scale(X_orig, axis=0, with_mean=True, with_std=True, copy=True)

让我们看看 Z-score 归一化所涉及的步骤。下图逐步展示了变换过程。

In [ ]:
mu     = np.mean(X_train,axis=0)   
sigma  = np.std(X_train,axis=0) 
X_mean = (X_train - mu)
X_norm = (X_train - mu)/sigma      

fig,ax=plt.subplots(1, 3, figsize=(12, 3))
ax[0].scatter(X_train[:,0], X_train[:,3])
ax[0].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[0].set_title("unnormalized")
ax[0].axis('equal')

ax[1].scatter(X_mean[:,0], X_mean[:,3])
ax[1].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[1].set_title(r"X - $\mu$")
ax[1].axis('equal')

ax[2].scatter(X_norm[:,0], X_norm[:,3])
ax[2].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[2].set_title(r"Z-score normalized")
ax[2].axis('equal')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle("distribution of features before, during, after normalization")
plt.show()

上图展示了训练集中的两个参数“房龄”和“平方英尺”之间的关系。*绘制它们时采用了相同的尺度*。
- 左图：未归一化。“面积（平方英尺）”特征的取值范围或方差远大于房龄特征
- 中图：第一步从每个特征中减去均值或平均值，使各特征都以零为中心。“房龄”特征的变化很难看出来，但“面积（平方英尺）”显然已经位于零附近。
- 右图：第二步除以方差，使两个特征都以零为中心，并具有相近的尺度。

让我们对数据进行归一化，并与原始数据进行比较。

In [ ]:
# normalize the original features
X_norm, X_mu, X_sigma = zscore_normalize_features(X_train)
print(f"X_mu = {X_mu}, \nX_sigma = {X_sigma}")
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X_train,axis=0)}")   
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X_norm,axis=0)}")

归一化将每列的峰峰值范围从相差数千倍缩小到相差 2–3 倍。

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3))
for i in range(len(ax)):
    norm_plot(ax[i],X_train[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count");
fig.suptitle("distribution of features before normalization")
plt.show()
fig,ax=plt.subplots(1,4,figsize=(12,3))
for i in range(len(ax)):
    norm_plot(ax[i],X_norm[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count"); 
fig.suptitle(f"distribution of features after normalization")

plt.show()

请注意，上面归一化数据的取值范围以零为中心，大致位于 +/- 1 之间。最重要的是，每个特征的取值范围都很相近。

让我们使用归一化数据重新运行梯度下降算法。
请注意 **alpha 的值大幅增大**。这会加快下降速度。

In [ ]:
w_norm, b_norm, hist = run_gradient_descent(X_norm, y_train, 1000, 1.0e-1, )

经过缩放的特征能**快得多、快得多地**得到非常准确的结果！请注意，经过这次相当短的运行后，每个参数的梯度都已经非常小。对于归一化特征的回归，学习率 0.1 是一个不错的起点。
让我们绘制预测值与目标值的对比图。请注意，预测使用归一化后的特征，而图中显示的是原始特征值。

In [ ]:
#predict target using normalized features
m = X_norm.shape[0]
yp = np.zeros(m)
for i in range(m):
    yp[i] = np.dot(X_norm[i], w_norm) + b_norm

    # plot predictions and targets versus original features    
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:,i],yp,color=dlorange, label = 'predict')
ax[0].set_ylabel("Price"); ax[0].legend();
fig.suptitle("target versus prediction using z-score normalized model")
plt.show()

结果看起来不错。需要注意以下几点：
- 使用多个特征时，无法再用一幅图展示结果与所有特征之间的关系。
- 生成图形时使用了归一化特征。凡是使用从归一化训练集学得参数进行的预测，也必须先进行归一化。

**预测**
生成模型的目的是用它预测数据集中没有的房屋价格。让我们预测一套面积为 1200 平方英尺、有 3 间卧室、1 层楼、房龄 40 年的房屋价格。回顾一下，你必须使用训练数据归一化时得到的均值和标准差对数据进行归一化。

In [ ]:
# First, normalize out example.
x_house = np.array([1200, 3, 1, 40])
x_house_norm = (x_house - X_mu) / X_sigma
print(x_house_norm)
x_house_predict = np.dot(x_house_norm, w_norm) + b_norm
print(f" predicted price of a house with 1200 sqft, 3 bedrooms, 1 floor, 40 years old = ${x_house_predict*1000:0.0f}")

**代价等高线**  
<img align="left" src="./images/C1_W2_Lab06_contours.PNG"   style="width:240px;" >另一种理解特征缩放的方法是从代价等高线的角度来看。当特征尺度不匹配时，在等高线图中绘制代价相对于参数的变化，图形会呈现不对称。

在下图中，参数的尺度是匹配的。左图是特征归一化之前，w[0]（平方英尺）与 w[1]（卧室数量）的代价等高线图。该图极不对称，构成完整等高线的曲线甚至不可见。相比之下，特征归一化后，代价等高线对称得多。因此，在梯度下降期间更新参数时，每个参数都能取得均衡的进展。

In [ ]:
plt_equal_scale(X_train, X_norm, y_train)


## 恭喜！
在本实验中，您：
- 使用了先前实验中开发的多特征线性回归例程；
- 探索了学习率 $\alpha$ 对收敛的影响；
- 发现了使用 z-score 归一化进行特征缩放对加快收敛的价值。

## 致谢
房屋数据来自 Dean De Cock 为数据科学教育整理的 [Ames 房屋数据集](http://jse.amstat.org/v19n3/decock.pdf)。